# RC Celta Seat Availability Alert Platform
### Project walkthrough — end-to-end technical & operational guide

**Repo:** [github.com/inakigorostiza/rc-celta-seat-availability-alert-platform](https://github.com/inakigorostiza/rc-celta-seat-availability-alert-platform)  
**Live landing:** [inakigorostiza.github.io/rc-celta-seat-availability-alert-platform](https://inakigorostiza.github.io/rc-celta-seat-availability-alert-platform/)  
**Tech demo video:** below


> **What you'll see in this notebook**
> 1. A working lead-recovery platform that integrates ticketing (ONEBOX) → CRM (Mailchimp) → transactional email (Mandrill) via a Supabase backend and an n8n orchestrator.
> 2. The architectural decisions and trade-offs behind each layer.
> 3. The legal & analytical scaffolding (GA4, RGPD / LOPDGDD / LSSI-CE) needed to run it in a club's real-world environment.


---
## 1 · Introduction & Vision

**Problem.** A non-trivial share of RC Celta supporters try to buy seats for high-demand matches while inventory shows *sold out*. A percentage of those seats free up later (returns, payment failures, releases from pre-allocations) but the fans who wanted them are already gone. This demand is silently lost.

**Solution.** A recovery platform that:

1. Captures interest from fans while there's no inventory — *"avísame si se libera un asiento en Pista – B1"*.
2. Continuously watches ONEBOX availability and detects the moment a grada has seats again.
3. Notifies every interested fan with a personalised email that deep-links straight into ONEBOX's seat-selector with full UTM attribution.

**Business value.** Recovered purchases from a segment that would otherwise churn, a clean first-party opt-in list segmented by event × grada (a goldmine for future match-day marketing), and full-funnel analytics from email open → ticket conversion.


### Demo video

Click play below to watch the walkthrough (open this notebook in Google Colab to play it inline; on GitHub render the video opens on YouTube).


In [ ]:
# Play the walkthrough inline (only works in Google Colab / Jupyter)
from IPython.display import YouTubeVideo
YouTubeVideo('-FURuRfDk8I', width=800, height=450)


Direct link: [https://youtu.be/-FURuRfDk8I](https://youtu.be/-FURuRfDk8I)


---
## 2 · Technology Stack

Deliberately few moving parts. Every component is either serverless, free-tier-friendly, or already in the club's toolbelt.

| Layer | Choice | Why |
|---|---|---|
| **Frontend** | Static HTML generated from the ONEBOX API | Zero runtime dependencies → hostable on GitHub Pages, Netlify, Vercel, S3. Build-time data avoids exposing the ticketing API to the browser. |
| **Backend API** | Supabase Edge Function (Deno) | Single-concern HTTPS endpoint; secrets stay server-side; Supabase client SDK with service-role access. |
| **Database** | Supabase Postgres (with Vault) | Encrypted secret storage, Row Level Security, built-in auth. Serves as the source of truth for the signup list. |
| **Ticketing** | ONEBOX TDS (PRE environment) | Existing RC Celta ticketing platform. Authoritative for session inventory. |
| **Marketing list** | Mailchimp | Where the club's marketing team already works. Tags + merge fields map 1-to-1 to our event × grada model. |
| **Transactional email** | Mandrill | Same login as Mailchimp, dedicated for one-to-one messages. Sub-second deliverability, per-email open/click analytics. |
| **Automation** | n8n | Visual workflow for ops; same instance can scale to cron-driven renewals, ticket-office reports, etc. |
| **Analytics** | Google Analytics 4 | Standard for marketing. `generate_lead` recommended event covers the core KPI. |
| **Hosting** | GitHub Pages | Free public HTTPS, automatic deploy on push. |


### Architecture diagram

```
┌── Landing page ──────────────────────────────────────────────┐
│  Static HTML · Celta branding · GA4 page_view                │
│  Event & sector data baked in at build time from ONEBOX       │
└───────────────┬──────────────────────────────────────────────┘
                │ POST {email, first_name, event_id, grada_id, consents, UTMs}
                ▼
┌── Supabase Edge Function  /functions/v1/avisame-signup ─────┐
│  · validates consent + lengths (defence in depth)            │
│  · INSERT into public.avisame_signups  (service role → RLS) │
│  · PUT  mailchimp /members/{md5(email)}  — merge_fields     │
│  · POST mailchimp /members/{hash}/tags   — event:X, grada:Y │
└───────────────┬──────────────────────────────────────────────┘
                │ every minute / hour / on demand
                ▼
┌── n8n workflow (manual + cron) ──────────────────────────────┐
│  ONEBOX /oauth/token → /sessions → /sessions/{id}/availability│
│  for each grada with seats:                                  │
│    Mailchimp Members filtered by tags event:X AND grada:Y     │
│    Mandrill /messages/send.json (branded HTML + UTMs)         │
└───────────────────────────────────────────────────────────────┘
```


---
## 3 · Project Roadmap & Milestones

Built in six incremental, independently verifiable phases. Every phase shipped something demo-able.

| Phase | Goal | Deliverable | Proof of work |
|---|---|---|---|
| **1 · Discovery** | Confirm ONEBOX credentials work against PRE; map the catalog data model. | `test-onebox.js` probe | Token acquired, `/catalog-api/v1/sessions` returns `metadata.total = 21`. |
| **2 · Data capture** | Build the signup form; pipe rows to Supabase. | `landing.html` + `avisame_signups` table with RLS | First real row lands via the form. |
| **3 · Marketing sync** | Mirror signups into Mailchimp with merge fields + tags. | Edge Function `avisame-signup` + Vault secrets | Contact appears in Mailchimp with all 12 merge fields + 6 tags populated. |
| **4 · Notifier** | Detect availability → email affected leads. | `n8n/avisame-workflow.json` | End-to-end run sends a branded email with deep-link + UTMs. |
| **5 · Analytics & legal** | Instrument GA4; add consent + legal guardrails. | `generate_lead` event + RGPD-aware consent columns | DebugView shows the event; RLS rejects signups without valid consents. |
| **6 · Deploy** | Make it public, reproducible, open for review. | GitHub Pages + public repo | Live URL returns 200 with landing page rendered. |


---
## 4 · Secrets, Credentials & Environment Management

Philosophy: **every secret lives in exactly one place**, and the place is picked so that rotation is a single edit.

| Credential | Single source of truth | Exposed to browser? | Rotation cost |
|---|---|---|---|
| ONEBOX `client_secret` | `.env` (local) + Supabase Vault (prod) | No | 1 file or 1 SQL row |
| Supabase service-role key | Supabase-injected `Deno.env` in the Edge Function | No | N/A (Supabase rotates) |
| Supabase anon key (JWT) | Public — baked into `landing.html` | **Yes, by design** | Regenerate landing.html |
| Mailchimp API key | Supabase Vault | No (never reaches browser) | 1 SQL row |
| Mailchimp audience ID | Supabase Vault | No | 1 SQL row |
| Mandrill API key | n8n Vars node (demo); real prod → n8n credential store | No | 1 node field |
| GA4 measurement ID | `generate-landing.js` constant | **Yes, by design** | Regenerate landing.html |


### `.env` layout (gitignored, never leaves local)

```bash
ONEBOX_CLIENT_SECRET=•••••••••••••••••••••••••••••••••
ONEBOX_CHANNEL_ID=2287

SUPABASE_URL=https://<project-ref>.supabase.co
SUPABASE_ANON_KEY=eyJhbGciOiJ...              # public JWT
```

### Secrets inside Supabase Vault (encrypted at rest)

```sql
select vault.create_secret('<mc-key>',        'mailchimp_api_key');
select vault.create_secret('<list-id>',       'mailchimp_audience_id');
select vault.create_secret('us1',             'mailchimp_server');
```

The Edge Function reads them through a `security definer` RPC limited to `service_role`, so no browser session can ever read them.


---
## 5 · ONEBOX TDS — Platform Overview & APIs

**ONEBOX TDS** is the ticketing distribution system RC Celta runs. For this platform we use three REST endpoints.

### Data model (what ONEBOX calls what)

```
event          (e.g. "Celta – Athletic · J29")
  └── session  (a specific performance of the event — a match kickoff)
        ├── price_types  (PISTA, VIP, PISO 1…  pricing + capacity buckets)
        └── sectors      (physical stadium zones — what fans call "grada")
              └── each sector has an availability per price_type
```

In user-facing copy we use **Partido** (= session / event) and **Grada** (= sector).


### Authentication — `seller-channel-client` OAuth2

| | |
|---|---|
| Endpoint | `POST https://api.oneboxtds.net/oauth/token` |
| Body | `grant_type=client_credentials`, `client_id=seller-channel-client`, `client_secret=<key>`, `channel_id=<id>` |
| Scope granted | `api-channels-all api-distribution-all api-catalog-all api-external-all api-gateway api-customers-mgmt-all api-orders-mgmt-all` |
| TTL | 12 hours (`expires_in = 43200`) |

Token is held in memory for a workflow run; no persistence needed.


### Catalog endpoints we use

| Verb · Path | Purpose | Key fields we read |
|---|---|---|
| `GET /catalog-api/v1/sessions` | List every session visible to the channel | `data[].id`, `data[].event.id`, `data[].event.name`, `data[].on_sale`, `data[].sold_out` |
| `GET /catalog-api/v1/sessions/{id}/availability` | Real-time availability for a session | `availability.available/total`, `price_types[]`, `sectors[]` with `price_types[].availability.available` |

### Purchase deep-link (used in the notification emails)

`https://tickets.oneboxtds.com/rccelta/select/{sessionId}?viewCode=V_blockmap&utm_source=avisame&utm_medium=email&utm_campaign=recovery-app&utm_content=event-{eventId}-grada-{gradaId}`

The `viewCode=V_blockmap` is functional (opens the seat map). The four UTMs let us attribute conversions back to the specific email that triggered the click.


---
## 6 · Landing Page — Design & ONEBOX Integration

**Design brief.** Clone the visual language of [avisame-celta.netlify.app](https://avisame-celta.netlify.app) (Celta celeste + red, watermark crest, ABANCA-style footer) so it feels native to the club.

**Technical brief.** Static HTML, no build framework, fetched catalog data baked in at compile time.


### Build pipeline

```bash
node generate-landing.js
# → authenticates against ONEBOX
# → pulls session 240895 + its 82 sectors
# → renders landing.html with the Grada dropdown populated
#   directly from the availability response
```

### What ends up in the browser

| Section | Data source |
|---|---|
| Hero image / crest / brand copy | Static, checked-in assets |
| Event recap card (name, date, availability) | Baked from `GET /sessions` + `/availability` at build time |
| Grada dropdown | Sectors enumerated from the availability response, sorted alphabetically |
| Consent checkbox | Required — form disabled until ticked |
| GA4 snippet | Hardcoded measurement ID `G-QFVXMME2ZY` |

### Form flow

1. User fills `email / first_name / last_name / grada / consent`.
2. JS captures `utm_*`, `referrer`, `navigator.userAgent`, `navigator.language`.
3. `POST {supabase_url}/functions/v1/avisame-signup` with the combined payload.
4. On 200 → show the confirmation panel, fire GA4 `generate_lead`.

**Live URL:** [https://inakigorostiza.github.io/rc-celta-seat-availability-alert-platform/](https://inakigorostiza.github.io/rc-celta-seat-availability-alert-platform/)


---
## 7 · Mailchimp — Audience Strategy & Integration

**Role.** Centralised, marketing-team-friendly audience. Every signup lands here so the club can run segmented campaigns (renewals, special offers, after-the-fact surveys) on top of the same list that powers the notifier.

**Audience:** `RC Celta – Avisos de entradas`


### Merge fields (what the marketing team sees per contact)

| Tag | Type | Label | Populated from |
|---|---|---|---|
| `FNAME` | Text | First Name | form |
| `LNAME` | Text | Last Name | form |
| `CHANNEL` | Text | Canal ONEBOX | `2287` |
| `LOCALE` | Text | Idioma | `navigator.language` |
| `SRC` | Text | Origen (utm_source) | query string |
| `CAMPAIGN` | Text | Campaña (utm_campaign) | query string |
| `LANDING` | Text | Landing | `avisame-celta` |
| `SIGNUPAT` | Date | Fecha del último alta | `new Date()` |
| `EVENT_ID` | Number | Evento (ID) | ONEBOX event |
| `EVENT_NAME` | Text | Evento | ONEBOX event |
| `SESSION_ID` | Number | Sesión (ID) | ONEBOX session |
| `GRADA_ID` | Number | Grada (ID) | ONEBOX sector |
| `GRADA_NAME` | Text | Grada | ONEBOX sector |
| `GRADA_CODE` | Text | Grada (código) | ONEBOX sector |


### Tags (for segmentation)

Each signup gets these tags stacked:

- `event:{event_id}` — e.g. `event:4587`
- `session:{session_id}` — e.g. `session:240895`
- `grada:{grada_id}` — e.g. `grada:224549`
- `grada-name:{grada_name}` — e.g. `grada-name:Pista - B1`
- `source:avisame` — constant, identifies the signup source
- `consent-marketing` — only if the fan opted in for marketing

**Why tags and not just more merge fields?** One fan can sign up for multiple grada × event combinations over a season. Tags **stack** — merge fields overwrite. Tags preserve the full subscription history; merge fields reflect the most recent intent.

### Sync mechanism

Supabase Edge Function calls Mailchimp directly:

- `PUT /lists/{audience_id}/members/{md5(lower(email))}` with `status_if_new=subscribed` → idempotent upsert.
- `POST /lists/{audience_id}/members/{hash}/tags` → accumulative tag attach.


---
## 8 · Mandrill — Transactional Email Delivery

**Role.** One-to-one, real-time notifications triggered by the n8n workflow. Not used for batch marketing.

**Why a separate service from Mailchimp.** Mailchimp is tuned for list blasts; Mandrill is tuned for transactional latency, per-message tracking, and DKIM/SPF per sending domain. Both live under the same Mailchimp login, so ops overhead is minimal.


### Sender-domain requirement

Mandrill refuses to send (`reject_reason: unsigned`) unless the `from_email`'s domain is configured as a signing domain on the account. For the demo we use `avisame@lin3s.com` (already verified with DKIM + SPF). For the real rollout:

1. Add `rccelta.es` in Mandrill → *Settings → Sending Domains*.
2. Publish the DKIM + SPF TXT records at the DNS level.
3. Verify in Mandrill. Then `entradas@rccelta.es` is fair game as a sender.

### Email template (what the fan sees)

- Celta-branded header (sky blue + red accent, crest reference).
- H1: *"¡Hay entradas!"*
- Personalised greeting.
- Recap card: **Partido · Grada · Disponibilidad (N localidades)**.
- Prominent red CTA button → ONEBOX seat selector with UTMs.
- Fallback text URL (copy-paste) for clients that block buttons.
- Footer linking back to the landing page.

### Tracking knobs enabled per send

| Setting | Value | Purpose |
|---|---|---|
| `track_opens` | `true` | Measure open rate per event / grada |
| `track_clicks` | `true` | Measure CTR on the "Comprar entrada" button |
| `tags` | `['avisame', 'event-{id}', 'grada-{id}']` | Group in Mandrill analytics for per-event / per-grada views |


---
## 9 · n8n — Notifier Workflow

**Workflow name:** *Avisame – ONEBOX → Mailchimp → Mandrill*  
**Trigger:** manual today, switchable to cron (every 5 – 15 minutes) for production.

### 12-node pipeline

| # | Node | Type | Purpose |
|---|---|---|---|
| 1 | **Manual Trigger** | trigger | Execute-on-demand entrypoint |
| 2 | **Vars** | Set | Central config: event_id, credentials, Mandrill defaults, purchase URL template |
| 3 | **Mailchimp Members** | built-in Mailchimp | Pull every subscribed member of the audience |
| 4 | **Aggregate members** | built-in Aggregate | Collapse to a single item holding the full list |
| 5 | **ONEBOX Auth** | HTTP | POST `/oauth/token` → extract bearer |
| 6 | **ONEBOX Sessions** | HTTP | GET `/sessions` with the bearer |
| 7 | **Filter by event_id** | Code | Keep only the sessions for the target event |
| 8 | **ONEBOX Availability** | HTTP | GET `/sessions/{id}/availability` per session |
| 9 | **Gradas with availability** | Code | Flatten to one item per sector with `available > 0` |
| 10 | **Find leads** | Code | Cross-reference aggregated members with gradas by tag |
| 11 | **Build Mandrill message** | Code (optional) | Assemble the Mandrill JSON payload |
| 12 | **Mandrill Send** | built-in Mandrill | POST `/messages/send.json` per lead |


### Evolution backlog (not in the MVP)

- **`notified_at` update** — after a successful send, UPDATE the corresponding row in `avisame_signups` so a subsequent run doesn't re-notify the same fan.
- **Cron trigger** — Schedule node firing every 10 min during high-demand windows (game week), downgraded off-season.
- **Split into two workflows** — a *Finder* (ONEBOX → availability) and a *Notifier* (Mailchimp → Mandrill), connected by an *Execute Sub-workflow* node, so the Notifier half can be reused for other triggers (e.g. early-bird campaigns).
- **Slack / WhatsApp fallback** — for ultra-high-demand gradas, add a second output to an internal ops channel for same-minute visibility.


---
## 10 · GitHub — Source Control & Pages Hosting

### Repository

- **URL:** [https://github.com/inakigorostiza/rc-celta-seat-availability-alert-platform](https://github.com/inakigorostiza/rc-celta-seat-availability-alert-platform)
- **Visibility:** public (architecture is shareable; secrets live outside the repo)
- **Branch model:** single `main` branch — small codebase doesn't warrant flows yet

### What's versioned vs not

| Tracked | Not tracked (.gitignored) |
|---|---|
| `test-onebox.js`, `generate-*.js`, `landing.html`, `index.html`, `supabase/`, `n8n/`, `assets/`, `README.md` | `.env` (secrets), `.claude/` + `.mcp.json` (IDE config with secret references), `sessions.html` + `examples.html` (large generated artifacts) |

### GitHub Pages hosting

- **Source:** `main` branch, root directory
- **Default document:** `index.html` (mirror of `landing.html`)
- **URL:** [https://inakigorostiza.github.io/rc-celta-seat-availability-alert-platform/](https://inakigorostiza.github.io/rc-celta-seat-availability-alert-platform/)
- **HTTPS:** enforced (GitHub-issued cert, auto-renewed)
- **Deploy time:** ~30 – 60 s per `git push`

### Secret hygiene executed before the first push

- `.env` gitignored from commit #0.
- `n8n/avisame-workflow.json` scrubbed: real ONEBOX secret + Mandrill key + Mailchimp list ID replaced with `REPLACE_WITH_*` placeholders.
- `.claude/settings.local.json` (contained a shell-history reference to ONEBOX secret) gitignored.
- Staged diff grep'd for every known secret string before `git commit`.


---
## 11 · Measurement Plan in GA4

**Measurement ID:** `G-QFVXMME2ZY` (baked into `landing.html` at build time)

### Events tracked

| Event | Source | Purpose |
|---|---|---|
| `page_view` | gtag.js auto | Traffic + source attribution via UTMs |
| `scroll`, `click`, `form_start`, `form_submit` | Enhanced Measurement | Engagement / funnel analysis |
| **`generate_lead`** | Custom — fired in the landing's submit success handler | **Core KPI — one conversion per successful signup** |


### `generate_lead` parameter map

| Parameter | Example | Register as custom dimension? |
|---|---|---|
| `method` | `avisame_form` | No (constant) |
| `event_id` | `4587` | ✅ Event scope |
| `event_name` | `Celta – Athletic · J29` | ✅ Event scope |
| `session_id` | `240895` | ✅ Event scope |
| `grada_id` | `224549` | ✅ Event scope |
| `grada_name` | `Pista - B1` | ✅ Event scope |
| `is_duplicate` | `false` | ✅ Event scope |
| `locale` | `es-ES` | Optional |

Custom dimensions are set up in **GA4 → Admin → Custom definitions**. Without registering them, the data is still collected but can't be used as a breakdown in the UI.


### KPIs & dashboards

| KPI | Computed from |
|---|---|
| **Signup rate** | `generate_lead` / `page_view` per session |
| **Signups per event** | `generate_lead` grouped by `event_id` |
| **Top-demand gradas** | `generate_lead` grouped by `grada_name` |
| **Source attribution** | `generate_lead` grouped by `session_source` / `session_campaign` |
| **Recovery rate** | Ticket purchases with `utm_source=avisame` ÷ notifications sent (Mandrill analytics) |
| **Funnel abandonment** | `form_start` vs `form_submit` vs `generate_lead` |

### Actions required in the GA4 admin

1. **Mark `generate_lead` as a Key Event** (counts as conversion + flows to Google Ads).
2. **Create custom dimensions** for the four event-scope params marked above.
3. **Enable Enhanced Measurement → Form interactions** on the web stream.
4. **Link GA4 ↔ Google Ads** if the club runs paid campaigns.
5. **Link GA4 ↔ BigQuery export** (optional) for long-tail analysis beyond GA4's UI limits.


---
## 12 · Legal & Privacy Compliance (RGPD · LOPDGDD · LSSI-CE)

### Regulatory landscape (Spain + EU)


#### RGPD — *Reglamento General de Protección de Datos* (UE 2016/679)

Normativa europea vigente que establece el marco general sobre cómo se deben recoger, tratar y almacenar los datos personales. Aplica a cualquier tratamiento de datos de residentes de la UE, independientemente de dónde opere el responsable.

**Obligaciones clave para esta plataforma:**

- Consentimiento **libre, específico, informado e inequívoco** antes de recoger el email (la casilla no puede venir premarcada).
- **Minimización de datos** — solo se recogen email + nombre + apellidos + grada + consentimientos + atribución básica. No se pide DNI, dirección, teléfono, etc.
- **Limitación de la finalidad** — los datos solo se usan para avisar de disponibilidad y, si el usuario lo acepta, para marketing.
- **Derechos ARCO-POL** (acceso, rectificación, cancelación, oposición, portabilidad, limitación) garantizados contractualmente.
- **Registro de actividades de tratamiento** documentado en un DPA (Data Processing Agreement).


#### LOPDGDD — *Ley Orgánica 3/2018 de Protección de Datos Personales y garantía de los derechos digitales*

Adaptación española del RGPD que refuerza los derechos de los usuarios y establece el régimen sancionador nacional. Aporta, entre otros:

- Edad mínima **14 años** para consentir por sí mismos el tratamiento (por debajo se requiere consentimiento parental).
- Derechos digitales específicos: portabilidad, al olvido en redes sociales, desconexión digital.
- Sanciones económicas escalonadas hasta **20 M€ o el 4 % de la facturación global anual** (la mayor de las dos).
- Obligación de designar un Delegado de Protección de Datos (DPD/DPO) en entidades que traten datos a gran escala.


#### LSSI-CE — *Ley 34/2002 de Servicios de la Sociedad de la Información y de Comercio Electrónico*

Regula el envío de comunicaciones comerciales electrónicas y el uso de cookies. Esta ley es la que **obliga específicamente al *opt-in* previo** para enviar emails comerciales a alguien que no sea cliente previo.

**Obligaciones clave:**

- Consentimiento **previo y expreso** para comunicaciones comerciales (artículo 21).
- **Mecanismo de baja gratuito y sencillo** en cada comunicación (enlace de unsubscribe en todos los emails).
- Identificación clara de la entidad remitente.
- **Banner de cookies** para cualquier cookie no estrictamente necesaria (GA4 cae aquí → se requiere aceptación antes de cargar gtag.js).
- Sanciones: hasta **150 000 €** por infracción grave.


### Mapeo normativa → implementación en la plataforma

| Requisito legal | Cómo se cumple hoy | Gap pendiente |
|---|---|---|
| Consentimiento inequívoco (RGPD art. 7) | Checkbox obligatorio `consent_privacy` + `consent_notify` antes de poder enviar | — |
| Consentimiento separado para marketing (LSSI-CE art. 21) | Columna `consent_marketing` independiente del consentimiento funcional | Demo los marca juntos; añadir checkbox separado visible en UI |
| Información previa (RGPD art. 13) | Enlace "política de privacidad" junto al checkbox | El enlace apunta a `#` — **falta redactar la política** |
| Minimización (RGPD art. 5.1.c) | Solo se recogen campos estrictamente necesarios | — |
| Limitación de finalidad (RGPD art. 5.1.b) | Datos usados solo para aviso + (opcional) marketing; RLS impide otros accesos | — |
| Seguridad (RGPD art. 32) | TLS en tránsito + encriptación en reposo (Supabase + Mailchimp + Mandrill); secretos en Vault | Completar ENS + pentest antes de producción |
| Derecho de supresión / baja (RGPD art. 17, LSSI-CE art. 22) | Columna `unsubscribed_at` + futuro endpoint público de baja | **Falta implementar el enlace de baja en los emails** |
| Portabilidad (RGPD art. 20) | La propia fila en `avisame_signups` + exportable vía Mailchimp | Formalizar un endpoint de autoservicio |
| Banner de cookies (LSSI-CE) | — | **Falta**: bloquear `gtag.js` hasta que el usuario acepte analítica |
| DPA con encargados (RGPD art. 28) | Contratos estándar Supabase, Mailchimp, Mandrill, n8n (si self-hosted, no aplica) | Revisar y firmar por el club |
| Registro de actividades (RGPD art. 30) | — | Redactar antes del go-live |
| DPO designado (LOPDGDD) | — | Designar y publicar contacto |


### Checklist antes de producción

- [ ] Redactar y publicar **política de privacidad** enlazada desde el formulario.
- [ ] Redactar y publicar **aviso legal** (identificación de la entidad según LSSI-CE).
- [ ] Implementar **banner de cookies** (Cookiebot, Iubenda, OneTrust o propio) con bloqueo de GA4 hasta consentimiento.
- [ ] Implementar **enlace de baja** funcional en todos los emails (Mandrill inserta `{{unsubscribe}}` merge var).
- [ ] Separar en UI **consentimiento funcional** (indispensable) del **consentimiento marketing** (opcional).
- [ ] Firmar **DPAs** con Supabase, Mailchimp/Mandrill, n8n (si cloud).
- [ ] Completar el **Registro de Actividades de Tratamiento (RAT)** interno del club.
- [ ] Confirmar que los datos se almacenan en la **UE o países con adequacy decision** (Supabase ofrece región EU-West).
- [ ] Designar **DPO** y publicar su contacto.
- [ ] **Pentest** ligero previo al go-live.


---
## Cierre & Q&A

- **Demo en vivo:** [https://inakigorostiza.github.io/rc-celta-seat-availability-alert-platform/](https://inakigorostiza.github.io/rc-celta-seat-availability-alert-platform/) — pruébalo desde el móvil para ver el formulario responsive.
- **Repositorio con todo el código y el SQL de Supabase:** [https://github.com/inakigorostiza/rc-celta-seat-availability-alert-platform](https://github.com/inakigorostiza/rc-celta-seat-availability-alert-platform)
- **Flujo n8n importable:** `n8n/avisame-workflow.json` en el repo.
- **Video resumen:** [youtu.be/-FURuRfDk8I](https://youtu.be/-FURuRfDk8I)

¿Preguntas?
